## Extraction of Data from ERA5 Dataset (GEE) ##

Bands to query from ERA5:
1) temperature_2m (air temperature)
2) skin_temperature (surface temperature)
3) soil_temperature_level_1 (soil temp)
4) volumetric_soil_water_layer_1
5) total_evaporation_sum
6) total_precipitation_sum

In [1]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [2]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,02-01-2011,0
1,-26.861111,28.884722,03-01-2011,1
2,-26.450000,28.085833,03-01-2011,2
3,-27.671111,27.236944,03-01-2011,3
4,-27.356667,27.286389,03-01-2011,4


In [12]:
wq_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Latitude     9319 non-null   float64
 1   Longitude    9319 non-null   float64
 2   Sample Date  9319 non-null   object 
 3   id           9319 non-null   int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 291.3+ KB


In [15]:
## need to convert Sample date dd-mm-yyyy to yyyy-mm-dd

wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df['Sample Date']

0       2011-01-02
1       2011-01-03
2       2011-01-03
3       2011-01-03
4       2011-01-03
           ...    
9314    2015-12-23
9315    2015-12-23
9316    2015-12-23
9317    2015-12-23
9318    2015-12-31
Name: Sample Date, Length: 9319, dtype: object

In [25]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(10000), #add a 10km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features



In [26]:
bands = ['temperature_2m',
         'skin_temperature', 
         'soil_temperature_level_1',
         'volumetric_soil_water_layer_1',
         'total_evaporation_sum', 
         'total_precipitation_sum']


era5_collection = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(bands)

In [27]:
def extract_median_values(feat):
    collection = era5_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.median()) # reduce image collection into a single image

    era5_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 11132)
    
    return era5_col.first()

In [28]:
fc_mapped = fc.map(extract_median_values)

In [29]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="era5_csv_export",
    fileNamePrefix= "era5_features_training",
    fileFormat='CSV'
)
task.start()

In [30]:
era5_df = pd.read_csv("../data/era5_features_training.csv")

# Drop irrelevant columns
era5_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)

era5_df = era5_df.merge(wq_df, on='id', how='left')
era5_df.drop(columns=['id'], inplace=True)
era5_df

,skin_temperature_median,soil_temperature_level_1_median,temperature_2m_median,total_evaporation_sum_median,total_precipitation_sum_median,volumetric_soil_water_layer_1_median,Latitude,Longitude,Sample Date
0,307.919884,307.833471,300.198610,-0.000085,0.000001,0.008799,-28.760833,17.730278,2011-01-02
1,293.665109,294.393847,292.444443,-0.004103,0.007830,0.458929,-26.861111,28.884722,2011-01-03
2,293.972931,294.725525,292.828033,-0.004010,0.006964,0.393498,-26.450000,28.085833,2011-01-03
3,294.280160,295.166729,293.509254,-0.003895,0.007228,0.409018,-27.671111,27.236944,2011-01-03
4,294.414705,295.304085,293.890963,-0.003909,0.004980,0.437815,-27.356667,27.286389,2011-01-03
...,...,...,...,...,...,...,...,...,...
9314,296.964143,296.845932,294.993890,-0.003694,0.000979,0.214896,-27.527500,30.858056,2015-12-23
9315,298.236266,298.396220,296.191045,-0.003708,0.000289,0.318879,-26.861111,28.884722,2015-12-23
9316,303.466747,303.856392,300.240368,-0.000438,0.000210,0.114664,-26.984722,26.632278,2015-12-23
9317,303.877579,304.383245,300.782481,-0.000162,0.000014,0.110157,-27.935000,26.126667,2015-12-23


Note: Temperature values are in kelvin thats why they are so high.

In [31]:
era5_df.to_csv("../data/era5_features_training.csv")        # now ready to go for preprocessing

# Repeat for Validation Set

In [32]:
val_df = pd.read_csv("../data/submission_template.csv")
val_df['id'] = val_df.index
val_df

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,id
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN,0
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,1
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN,2
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,3
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...
195,-33.771111,25.386667,06-12-2012,NaN,NaN,NaN,195
196,-33.185361,27.390750,04-09-2014,NaN,NaN,NaN,196
197,-32.043333,27.822778,28-09-2015,NaN,NaN,NaN,197
198,-33.001667,25.161389,08-01-2015,NaN,NaN,NaN,198


In [33]:

features_val = []

for index, row in wq_df.iterrows():
    feat_val = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(10000), #add a 10km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features_val.append(feat_val)

fc_val = ee.FeatureCollection(features_val)

In [34]:
fc_mapped_val = fc_val.map(extract_median_values)

In [35]:
task = ee.batch.Export.table.toDrive(
    collection=fc_mapped_val,
    description="era5_csv_export",
    fileNamePrefix= "era5_features_training",
    fileFormat='CSV'
)
task.start()